# QK-Norm in Qwen3-0.6B

Qwen3 adds a normalization step that its Llama-family predecessors do not have:
the query and key vectors are RMS-normalized, per head, immediately before the
rotary embedding and the attention dot product. This is the single most visible
architectural difference between Qwen3's attention block and the Llama lineage
it otherwise resembles.

This notebook works out what that step does and what it means for building on it.
It moves from an intuition for where the normalization sits, to the measured
effect on a real Qwen3-0.6B forward pass, to the mathematics of RMSNorm derived
from scratch and checked against the model's own implementation, to the concrete
tensor shapes it runs on. Along the way three measurements probe the parts a
practitioner runs into: whether small logits mean spread-out attention, how the
normalizer weight survives low-precision quantization, and whether a logit
soft-cap would add anything on top.

Two supporting scripts hold the code this notebook inspects and runs:
`scripts/rmsnorm_mechanism.py` (the from-scratch derivation and its equivalence
proof) and `scripts/capture_qk_stats.py` (the instrumentation that produced the
figures), together with `scripts/quant_sensitivity.py` for the quantization
measurement. Figures are regenerated from `results/qk_stats.json` and
`results/quant_sensitivity.json`.

In [ ]:
import sys
from pathlib import Path

# Make the scripts importable whether the notebook is opened from the repo root
# or elsewhere.
REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

import inspect
import rmsnorm_mechanism as rms

## Where the normalization sits

A single Qwen3 decoder layer follows the usual pre-norm shape: normalize, run
the sub-layer, add the residual. Inside the attention sub-layer, the ordering
that matters here is

$$
\text{hidden} \;\rightarrow\; \text{QKV projection} \;\rightarrow\;
\boxed{\text{QK-Norm}} \;\rightarrow\; \text{RoPE} \;\rightarrow\;
\text{GQA} \;\rightarrow\; \text{SDPA} \;\rightarrow\; o_{\text{proj}}.
$$

The projection produces per-head query and key vectors. QK-Norm normalizes each
of those vectors, on its own, before any positional information is applied. Only
then does RoPE rotate them and the dot product combine them.

The object being normalized is one head's vector, of length $d_h$ (the head
dimension), not the full model-dimension hidden state. That distinction is the
whole point: the normalization is *per head*, and its learnable scale has
$d_h$ parameters, not $d_{\text{model}}$.

## RMSNorm, briefly

The normalizer is RMSNorm. For a vector $x \in \mathbb{R}^{d}$ with a learnable
per-dimension weight $w \in \mathbb{R}^{d}$,

$$
\operatorname{RMSNorm}(x) \;=\; \frac{x}{\sqrt{\dfrac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \;\odot\; w .
$$

Unlike LayerNorm, there is no mean subtraction and no bias: RMSNorm rescales by
the root-mean-square of the entries and nothing else. The original QK-Norm
proposal (Dehghani et al., 2023, in ViT-22B) used LayerNorm; Qwen3 uses RMSNorm,
applied on the head dimension. The consequences of that choice are what the rest
of this notebook makes concrete.

## What it does to a real forward pass

The reason to normalize $q$ and $k$ is that, without it, their magnitude grows
with depth. Deeper layers produce larger query and key vectors, and since the
attention logit is a dot product of the two, unnormalized growth feeds straight
into the logits.

The figure below measures the per-head vector norm of $q$ and $k$ at every one
of Qwen3-0.6B's 28 layers, over a single continuous 4096-token document, both
before and after the QK-Norm step. Before normalization, the norm climbs
monotonically with depth. After it, the same quantity sits in a narrow band.

![pre vs post norm across depth](assets/qknorm_prepost_depth.png)

The pre-norm curve spreads by roughly a factor of 88 across depth. The post-norm
curve narrows that to about 2.3x for queries. Keys stay somewhat looser (about
9.5x), which is not noise: the two normalizers have separate learned weights,
and the key normalizer's weight is larger. That asymmetry is worth holding onto.

The single-document view could be a quirk of one input. Repeating the
measurement over 200 independent MMLU-Redux prompts shows the compression is a
property of the model rather than the document: each layer's post-norm band is
tight regardless of the prompt, and the query/key asymmetry survives pooling.

![aggregate band across 200 samples](assets/qknorm_aggregate_band.png)

### Why this bounds the logits

With $q$ and $k$ normalized before the dot product, the attention logit

$$
z_{ij} \;=\; \frac{\langle q_i, k_j \rangle}{\sqrt{d_h}}
$$

is a dot product of two vectors whose norms are controlled. By Cauchy-Schwarz,
$|\langle q_i, k_j\rangle| \le \lVert q_i \rVert \, \lVert k_j \rVert$, so
bounding the norms bounds the logit directly. The often-cited motivation for
QK-Norm is avoiding FP16 overflow in the logits on long sequences. On
Qwen3-0.6B that overflow does not happen, and the figure below shows why it does
not: the largest attention logit across all 28 layers, over 4096 tokens, is a
rounding error against the FP16 ceiling.

The point is not that overflow is weakly evidenced here. It is that overflow is
*prevented*: the causal chain from large pre-norm magnitude, to small post-norm
magnitude, to small logits, is fully contained in the model's own data. A model
without QK-Norm, such as Qwen2.5, lacks this compression step and so does not
get this bound for free.

![logit magnitude vs FP16 ceiling](assets/qknorm_logit_ceiling.png)

### Small logits are not the same as spread-out attention

Bounding the logits is a statement about magnitude. It says nothing about
*concentration*: how sharply the softmax over those logits peaks. These are two
separate axes, and it is easy to slide from one to the other. A logit maximum of
47.8 is far from the FP16 ceiling, but it is more than enough to make a softmax
nearly one-hot, since a gap of about ten between the top logit and the rest
already puts over 99% of the probability mass on a single key. QK-Norm bounds
magnitude; it neither forces nor prevents a head from concentrating.

To see concentration directly, measure the attention entropy per head, per
layer:

$$
S(A_i) = -\sum_{j} A_{ij} \log A_{ij},
$$

where $A_{ij}$ is the softmax attention weight from query $i$ to key $j$. Low
entropy means the row collapsed toward one key; high entropy means it spread
across many. The maximum possible for a 4096-key row is $\log(4096) = 8.32$
nats. The capture computes this inside the same chunked pass that produced the
logit statistics, over the causally masked rows only:

In [ ]:
import capture_qk_stats as cap

# The per-row entropy is computed by _row_entropy, called inside the chunked
# _logit_stats pass. The function itself:
print(inspect.getsource(cap._row_entropy))

Two views of the result, both layer by head. The left is each head's mean
entropy over the run; the right is the minimum entropy it produced at any
qualifying position (rows with at least 32 attendable keys, so a low value
reflects a choice to be sharp rather than a short-context artifact). No health
threshold is imposed; read the spread directly.

![per-head attention entropy](assets/qknorm_entropy_heatmaps.png)

Entropy varies widely across heads and drifts lower with depth: later layers
hold many sharp heads, yet every layer also carries high-entropy heads spread
across the context. The two axes are visibly independent. Bounded logits and
concentrated attention coexist, which is exactly what "QK-Norm bounds magnitude,
not concentration" means in practice.

Returning to magnitude: one more thing the overflow worry would need is
magnitude growing with sequence position. If logits crept upward as the context
filled, a long enough sequence could still reach the ceiling. They do not.
Across the full 4096-token run the per-position logit maximum is flat after the
first few hundred tokens, at every depth.

![per-position logit max](assets/qknorm_per_position.png)

## The mechanism, derived and checked

The claims above rest on knowing exactly what RMSNorm computes. Rather than
trust the operation as a black box, we derive it from scratch and then prove the
derivation is numerically identical to the implementation the model actually
runs.

Writing the definition out step by step: cast to float32 for the statistic,
compute the mean of squares over the last axis, divide by its root, cast back,
then apply the weight.

$$
\mu_2 = \frac{1}{d}\sum_{i=1}^{d} x_i^2,
\qquad
\hat{x} = \frac{x}{\sqrt{\mu_2 + \epsilon}},
\qquad
y = w \odot \hat{x}.
$$

The float32 cast is not cosmetic. Qwen3RMSNorm computes $\mu_2$ and the
normalization in float32 and only then casts back to the input dtype. A
from-scratch version that stays in the input dtype diverges on half precision.
Here is the derivation, shown as the actual source that runs:

In [ ]:
print(inspect.getsource(rms.rms_norm_from_scratch))

And the upstream implementation, imported live from transformers, for
comparison:

In [ ]:
from transformers.models.qwen3.modeling_qwen3 import Qwen3RMSNorm
print(inspect.getsource(Qwen3RMSNorm.forward))

The derivation is only worth anything if it matches. The check below builds a
real `Qwen3RMSNorm`, gives it random weights (no model download needed), runs
both implementations on the same random input, and asserts they agree. A drift
guard first confirms the upstream source still performs the operations the
derivation replicates; if a future transformers version changes the mechanism,
the guard fails loudly rather than letting the derivation quietly go stale.

In [ ]:
result = rms.assert_rms_matches_qwen3()
print(f"max abs difference: {result['max_abs_diff']:.2e}")
print(f"over {result['n_tokens']} tokens x {result['dim']} dims")

### A property the derivation exposes: per-head scale invariance

RMSNorm divides by the root-mean-square of the vector. Multiplying the whole
vector by a constant multiplies that RMS by the same constant, so the ratio is
unchanged: RMSNorm is invariant to the overall scale of its input. What survives
normalization is the vector's *direction*, not its magnitude.

This has a direct consequence for the earlier figures. The post-norm band is
narrow but not perfectly flat, and it differs between queries and keys. Since
input magnitude normalizes away entirely, neither of those residual effects can
come from the input. They come from the learned weight $w$: the post-norm spread
is the imprint of $w$, and the query/key asymmetry is the difference between the
query normalizer's weight and the key normalizer's.

The demonstration below shows both facts on a per-head tensor: perturbing one
head's direction changes only that head's output (normalization is per head),
while scaling one head by 10x changes nothing (scale invariance).

In [ ]:
rms.demo_per_head_axis()

The same two facts are easier to see in motion. Below, one query vector is shown in a single RoPE rotation plane at every depth; the arrow lengths are the real per-layer mean head-norms, only the rotation plane is chosen for display. As depth advances the pre-norm arrow (gold) grows from about 4 to about 165, while the post-norm arrow (violet) stays near 25 throughout. At the first layer the post-norm arrow is the longer of the two, so normalization is not shortening the vector; it is fixing the length independently of depth. The pre-norm length tracks depth, the post-norm length tracks only the learned weight.

![depth sweep of q vector under RoPE](assets/qknorm_depth_sweep.gif)

### What quantizing that weight costs

If the post-norm length is set entirely by the learned weight, the weight is
where low-precision deployment will bite. The weight is a single 128-vector per
normalizer, shared across all heads, so quantizing it is a small operation with
a per-head consequence: the same rounded weight, applied through RMSNorm to each
head's different activations, produces a different output error per head.

The measurement quantizes `q_norm.weight` to INT8 and INT4 (symmetric
per-tensor, what an edge runtime uses for a vector this small) and reports both
the weight rounding error and, using real activations from a short forward pass,
the per-head output error. Symmetric per-tensor is deliberate: per-channel
quantization would fit each element independently and hide the per-head
sensitivity this is meant to surface. The quantizer itself:

In [ ]:
import quant_sensitivity as qs
print(inspect.getsource(qs.quantize_symmetric_per_tensor))

![quantization sensitivity](assets/qknorm_quant_sensitivity.png)

Left: the weight rounding error per layer. INT8 stays around a few percent
relative error; INT4 jumps to tens of percent. Right: the per-head output error
that INT4 induces. Across most layers the per-head spread is modest, roughly
1.2 to 1.5x between the least and most affected head, so the shared weight
degrades fairly uniformly. The first layer is the exception, with a wider spread
and the largest errors overall. Whatever makes layer 0 unusual here is the same
place the key normalizer's post-norm magnitude was most extreme earlier.

## The concrete shapes in Qwen3-0.6B

The derivation is dimension-agnostic; the model is not. In Qwen3-0.6B the query
and key normalizers operate on the head dimension $d_h = 128$, not the model
dimension $d_{\text{model}} = 1024$. The normalizer's weight is therefore a
128-vector, shared across all heads but applied to each head's vector
independently.

The guard below constructs a real `Qwen3Attention` on the meta device (structure
only, no parameter memory, no download) and reads back what it actually built:
that `q_norm` and `k_norm` exist, are `Qwen3RMSNorm`, and are sized to the head
dimension. This is the check the capture script runs before trusting any
statistic, reproduced here so the shape claim is verified rather than asserted.

In [ ]:
cap.assert_qknorm_module_shape()
print("q_norm / k_norm confirmed: Qwen3RMSNorm, sized to head_dim")

Putting the ordering and the shapes together, the operation each attention layer
performs on its query stream is

$$
q \;=\; \operatorname{reshape}\big(q_{\text{proj}}(h),\; (\,\cdot\,,\, H,\, d_h)\big),
\qquad
q \;\leftarrow\; \operatorname{RMSNorm}_{d_h}(q),
\qquad
q \;\leftarrow\; \operatorname{RoPE}(q),
$$

with the identical operation on $k$ using its own normalizer weight, and the
attention logits formed from the normalized, rotated $q$ and $k$. The
normalization happens on the reshaped, per-head view, before RoPE, which is
exactly the ordering the drift guard pins to the source.

## If bounding weren't enough: logit soft-capping

QK-Norm bounds the logits by bounding their inputs. A different tool bounds the
logits directly: Gemma-2-style soft-capping replaces each logit $z$ with

$$
z' = \text{cap} \cdot \tanh\!\left(\frac{z}{\text{cap}}\right),
$$

which can never exceed $\pm\text{cap}$ no matter how large $z$ grows. It is worth
knowing where this would act on Qwen3's actual logits, both to see whether it
would add anything and to understand that it works on a different part of the
range than QK-Norm does. The transform:

In [ ]:
import numpy as np

def soft_cap(z, cap):
    return cap * np.tanh(z / cap)

# Applied to the real captured logit extremes, per layer, at several caps.
import json
stats = json.loads((REPO_ROOT / "results" / "qk_stats.json").read_text())
layers = stats["long_run"]["layers"]
abs_max = max(layers[str(i)]["logits"]["abs_max"] for i in range(len(layers)))
p99 = max(layers[str(i)]["logits"]["p99_abs"] for i in range(len(layers)))

print(f"real logit p99     = {p99:.1f}")
print(f"real logit abs_max = {abs_max:.1f}")
for cap in (10, 20, 30, 50):
    print(f"  cap={cap:2d}: abs_max {abs_max:.1f} -> {soft_cap(abs_max, cap):.1f}"
          f"   p99 {p99:.1f} -> {soft_cap(p99, cap):.1f}")

![soft-cap transfer with real logit markers](assets/qknorm_softcap_transfer.png)

The markers place Qwen3's real logits on the input axis. The bulk of the
distribution, at and below the p99 of 26.3, sits in the near-linear region every
reasonable cap leaves almost untouched; only the extreme tail near the abs_max
of 47.8 falls into the bent region. Even a cap of 50 pulls that tail down by
about a fifth, because $\tanh$ curves well before its argument reaches the cap,
but it barely touches the mass of the distribution. Soft-cap and QK-Norm act on
different parts of the range: QK-Norm keeps the whole distribution controlled
through its inputs, while soft-cap clips the extreme tail of the output. On a
model whose logits QK-Norm already holds this low, soft-cap has little left to
do.

## Summary

QK-Norm RMS-normalizes each head's query and key vector, on the head dimension,
before RoPE and the dot product. On Qwen3-0.6B this compresses a roughly 88x
depth-wise spread in $q/k$ magnitude down to about 2.3x for queries, holds
across 200 independent prompts, and keeps every attention logit far below the
FP16 ceiling with no growth over a 4096-token sequence. Because RMSNorm is
scale-invariant, the magnitude of the incoming vectors is discarded entirely;
what remains is direction, shaped by the learned per-head weight, and the
query/key asymmetry in the post-norm band is the signature of those two separate
weights. The overflow that QK-Norm is meant to prevent does not occur here
precisely because the normalization is present.

Three measurements sharpen what that does and does not buy. Bounded logits are
not spread-out attention: entropy varies widely per head and falls with depth,
so magnitude and concentration are independent axes. The normalizer weight,
which carries the whole post-norm length, tolerates INT8 well but loses tens of
percent of relative precision at INT4, fairly uniformly across heads except a
more sensitive first layer. And a logit soft-cap, the direct alternative to
bounding by inputs, has little left to clip once QK-Norm has already held the
logits this low, touching only the extreme tail and leaving the bulk of the
distribution unchanged.

## References

Primary sources for the mechanism and the alternatives discussed above.

**QK-Norm, original proposal.** Dehghani et al. (2023), *Scaling Vision Transformers to 22 Billion Parameters*, arXiv:2302.05442. Introduces normalizing queries and keys before the attention dot product, using LayerNorm, to control attention-logit growth at scale. Qwen3 adopts the idea with RMSNorm on the head dimension.

**Attention-logit-growth instability.** Wortsman et al. (2024), *Small-scale proxies for large-scale Transformer training instabilities*, arXiv:2309.14322. Reproduces the logit-growth instability at small scale and confirms qk-layernorm as an effective mitigation. Used here as the public, readable account of the instability QK-Norm addresses.

**Logit soft-capping.** Gemma Team (2024), *Gemma 2: Improving Open Language Models at a Practical Size*, arXiv:2408.00118. Source of the `cap * tanh(z / cap)` logit soft-cap examined in the soft-cap section as the direct alternative to bounding logits through their inputs.

**Qwen3 architecture.** Qwen Team (2025), *Qwen3 Technical Report*, arXiv:2505.09388, and the reference implementation in HuggingFace Transformers (`transformers.models.qwen3.modeling_qwen3`). The exact `q_norm` / `k_norm` placement, head-dimension sizing, and forward ordering used throughout this notebook were verified directly against the Transformers source, not against the report alone.

RMSNorm itself is from Zhang and Sennrich (2019), *Root Mean Square Layer Normalization*, arXiv:1910.07467.